# 03 · Silver assignment ⭐
`bronze.per_all_assignments_m` → `silver.hcmassignment` (module 06 §2). Oracle's effective dates already carry history, so this is an upsert, not SCD2.

✅ Expect **2,705** rows · 2,693 primary · ACTIVE 2,703 / PENDING_REVIEW 2.

In [ ]:
from pyspark.sql import functions as F, Window
from hcm_utils import load_config, parse_date, to_int, latest_batch, issue_rows, write_issues, upsert

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("run_id", "manual")
catalog = dbutils.widgets.get("catalog")
run_id  = dbutils.widgets.get("run_id")
cfg = load_config()

SRC_FILE = "PER_ALL_ASSIGNMENTS_M"
REF = "ASSIGNMENT_ID"
COLS = ["ASSIGNMENT_ID", "PERSON_ID", "ORGANIZATION_ID", "JOB_ID", "GRADE_ID", "LOCATION_ID",
        "LEGAL_ENTITY_ID", "ASSIGNMENT_STATUS_TYPE", "ASSIGNMENT_CATEGORY", "PRIMARY_FLAG",
        "EFFECTIVE_START_DATE", "EFFECTIVE_END_DATE", "LAST_UPDATE_DATE"]
RAW_DATES = ["EFFECTIVE_START_DATE", "EFFECTIVE_END_DATE", "LAST_UPDATE_DATE"]
dom = cfg["domains"]["ASSIGNMENT_STATUS_TYPE"]
value_map = F.create_map(*[F.lit(x) for kv in dom["value_map"].items() for x in kv])

In [ ]:
# 1. STANDARDISE
df = (latest_batch(f"{catalog}.bronze.per_all_assignments_m").select(*COLS)
      .withColumn("EFF_START", parse_date("EFFECTIVE_START_DATE"))
      .withColumn("EFF_END",   parse_date("EFFECTIVE_END_DATE"))
      .withColumn("LAST_UPD",  parse_date("LAST_UPDATE_DATE"))
      .withColumn("STATUS_STD", F.upper(F.trim("ASSIGNMENT_STATUS_TYPE")))
      .withColumn("STATUS_FIX", F.coalesce(F.try_element_at(value_map, F.col("STATUS_STD")), F.col("STATUS_STD"))))

unparsed = df.filter("EFF_START IS NULL OR EFF_END IS NULL OR LAST_UPD IS NULL").count()
assert unparsed == 0, f"{unparsed} rows have unparseable dates - add a format to parse_date()"

legal_ok = (latest_batch(f"{catalog}.bronze.per_legal_employers")
            .select("LEGAL_ENTITY_ID").distinct().withColumn("_le_ok", F.lit(True)))
df = df.join(legal_ok, "LEGAL_ENTITY_ID", "left")

In [ ]:
# 2. DETECT (all rows)
is_iso = F.col("EFFECTIVE_START_DATE").rlike(r"^\d{4}-\d{2}-\d{2}$")
in_domain = F.col("STATUS_STD").isin(dom["allowed"])
issues = [
    issue_rows(df.filter(~is_iso), SRC_FILE, REF, "FORMAT_INCONSISTENCY", "EFFECTIVE_START_DATE in non-ISO format", "FIXED"),
    issue_rows(df.filter("GRADE_ID IS NULL"), SRC_FILE, REF, "NULL_VALUE", "GRADE_ID is null", "FIXED"),
    issue_rows(df.filter("LOCATION_ID IS NULL"), SRC_FILE, REF, "NULL_VALUE", "LOCATION_ID is null", "FIXED"),
    issue_rows(df.filter(F.col("ASSIGNMENT_STATUS_TYPE") != F.col("STATUS_STD")), SRC_FILE, REF,
               "CASING_WHITESPACE", "ASSIGNMENT_STATUS_TYPE needs trim+uppercase", "FIXED"),
    issue_rows(df.filter(~in_domain & F.col("STATUS_FIX").isin(dom["allowed"])), SRC_FILE, REF,
               "INVALID_VALUE", "ASSIGNMENT_STATUS_TYPE outside allowed domain (mapped)", "FIXED"),
    issue_rows(df.filter(~in_domain & ~F.col("STATUS_FIX").isin(dom["allowed"])), SRC_FILE, REF,
               "INVALID_VALUE", "ASSIGNMENT_STATUS_TYPE outside allowed domain", "FLAGGED"),
    issue_rows(df.filter("_le_ok IS NULL"), SRC_FILE, REF, "ORPHANED_FK",
               "LEGAL_ENTITY_ID does not exist in PER_LEGAL_EMPLOYERS", "FIXED"),
    issue_rows(df.filter("EFF_END < EFF_START"), SRC_FILE, REF, "CONSISTENCY_VIOLATION",
               "EFFECTIVE_END_DATE precedes EFFECTIVE_START_DATE", "FLAGGED"),
    issue_rows(df.filter("LAST_UPD < EFF_START"), SRC_FILE, REF, "STALE_TIMESTAMP",
               "LAST_UPDATE_DATE older than EFFECTIVE_START_DATE", "FLAGGED"),
]

In [ ]:
# 3. EXACT DEDUP (on normalised values, so format-variant re-sends count as exact)
BIZ = [c for c in COLS if c not in RAW_DATES] + ["EFF_START", "EFF_END", "LAST_UPD"]
df = df.withColumn("_rn", F.row_number().over(Window.partitionBy(*BIZ).orderBy(F.lit(1))))
issues.append(issue_rows(df.filter("_rn > 1"), SRC_FILE, REF, "DUPLICATE_ROW", "Exact re-send duplicate", "DROPPED"))
df = df.filter("_rn = 1").drop("_rn")

# 4. CHANGED RE-SENDS: keep latest
wk = Window.partitionBy("PERSON_ID", "ASSIGNMENT_ID", "EFF_START")
df = (df.withColumn("_cnt", F.count("*").over(wk))
        .withColumn("_rn", F.row_number().over(wk.orderBy(F.col("LAST_UPD").desc()))))
issues.append(issue_rows(df.filter("_cnt > 1 AND _rn > 1"), SRC_FILE, REF, "DUPLICATE_ROW_CHANGED",
                         "Re-send with updated values and later LAST_UPDATE_DATE", "DROPPED"))
df = df.filter("_rn = 1").drop("_rn", "_cnt")

In [ ]:
# 5. MULTI-ASSIGNMENT
per_person = df.groupBy("PERSON_ID").agg(
    F.countDistinct("ASSIGNMENT_ID").alias("n_asg"),
    F.sum(F.when(F.upper(F.trim("PRIMARY_FLAG")) == "Y", 1).otherwise(0)).alias("n_primary"))
issues.append(issue_rows(per_person.filter("n_asg > 1"), SRC_FILE, "PERSON_ID", "MULTI_ASSIGNMENT",
                         "Worker has concurrent assignment rows (1 primary, 1+ secondary)", "FLAGGED"))
# NB: with real multi-version history, check "one primary per person per point in time" instead
assert per_person.filter("n_primary != 1").count() == 0, "Some persons do not have exactly one primary assignment"

In [ ]:
# 6. FIX + 7. FLAG
grade_by_job = (df.filter("GRADE_ID IS NOT NULL").groupBy("JOB_ID", "GRADE_ID").count()
                .withColumn("_r", F.row_number().over(
                    Window.partitionBy("JOB_ID").orderBy(F.desc("count"), F.col("GRADE_ID"))))
                .filter("_r = 1").select("JOB_ID", F.col("GRADE_ID").alias("DEFAULT_GRADE_ID")))
df = df.join(grade_by_job, "JOB_ID", "left")

unknown_loc = cfg["null_handling"]["PER_ALL_ASSIGNMENTS_M"]["LOCATION_ID"]["value"]
dq_flags = F.array_compact(F.array(
    F.when(F.col("GRADE_ID").isNull(), F.lit("GRADE_DEFAULTED_BY_JOB")),
    F.when(F.col("LOCATION_ID").isNull(), F.lit("LOCATION_UNKNOWN")),
    F.when(F.col("_le_ok").isNull(), F.lit("ORPHANED_LEGAL_ENTITY")),
    F.when(~F.col("STATUS_FIX").isin(dom["allowed"]), F.lit("INVALID_STATUS")),
    F.when(F.col("EFF_END") < F.col("EFF_START"), F.lit("END_BEFORE_START")),
    F.when(F.col("LAST_UPD") < F.col("EFF_START"), F.lit("STALE_TIMESTAMP")),
))

silver = df.select(
    F.col("ASSIGNMENT_ID").cast("bigint").alias("AssignmentId"),
    F.col("PERSON_ID").cast("bigint").alias("PersonId"),
    to_int("ORGANIZATION_ID").alias("OrganizationId"),
    to_int("JOB_ID").alias("JobId"),
    F.coalesce(to_int("GRADE_ID"), to_int("DEFAULT_GRADE_ID")).alias("GradeId"),
    F.coalesce(to_int("LOCATION_ID"), F.lit(unknown_loc)).alias("LocationId"),
    F.when(F.col("_le_ok"), to_int("LEGAL_ENTITY_ID")).otherwise(F.lit(-1)).alias("LegalEntityId"),
    F.col("STATUS_FIX").alias("AssignmentStatus"),
    F.trim("ASSIGNMENT_CATEGORY").alias("AssignmentCategory"),
    (F.upper(F.trim("PRIMARY_FLAG")) == "Y").alias("IsPrimary"),
    F.col("EFF_START").alias("EffectiveStartDate"),
    F.col("EFF_END").alias("EffectiveEndDate"),
    F.col("LAST_UPD").alias("LastUpdateDate"),
    F.current_date().between(F.col("EFF_START"), F.col("EFF_END")).alias("IsCurrent"),
    dq_flags.alias("DqFlags"),
    F.lit(run_id).alias("RunId"),
    F.current_timestamp().alias("SilverLoadTimestamp"),
)

In [ ]:
# 8. WRITE
upsert(silver, f"{catalog}.silver.hcmassignment", keys=["AssignmentId", "EffectiveStartDate"])
print("issues logged:", write_issues(issues, catalog, run_id))
print("hcmassignment rows:", spark.table(f"{catalog}.silver.hcmassignment").count())